# 06 — Product Analysis
## Superstore Product Analytics

### Objetivo

Analisar a contribuição econômica dos produtos da Superstore, identificando diferenças de valor, concentração da receita e participação dos produtos nas transações de maior impacto.

A análise será realizada no nível de produto, complementando as etapas anteriores sem repetir as avaliações por categoria e subcategoria.

## 1. Configuração e Base Analítica

### 1.1 — Configuração do Ambiente e Carregamento dos Dados

A análise utiliza a camada de dados processados criada e validada nas etapas anteriores do projeto.

Nesta etapa, o objetivo é configurar o ambiente, localizar a fonte analítica e carregar os dados necessários para a construção da visão por produto.

In [16]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.metrics import calculate_top_concentration

PROCESSED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "sales_data_clean.parquet"
)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Fonte analítica: {PROCESSED_DATA_PATH}")
print(f"Arquivo disponível: {PROCESSED_DATA_PATH.exists()}")

Raiz do projeto: c:\superstore-analytics
Fonte analítica: c:\superstore-analytics\data\processed\sales_data_clean.parquet
Arquivo disponível: True


In [17]:
df = pd.read_parquet(PROCESSED_DATA_PATH)

print(f"Registros carregados: {len(df):,}")
print(f"Colunas disponíveis: {df.shape[1]}")

Registros carregados: 9,799
Colunas disponíveis: 19


### 1.2 — Construção da Visão por Produto

A base processada possui granularidade de item de pedido. Para analisar a contribuição econômica dos produtos, será construída uma visão agregada no nível de Product ID.

Para cada produto serão calculados:

- **Revenue:** receita total gerada pelo produto;
- **Orders:** número de pedidos distintos em que o produto esteve presente.

Essa estrutura será utilizada nas análises de valor econômico, concentração da receita e participação dos produtos em pedidos de maior impacto.

In [18]:
product_summary = (
    df.groupby(
        ["Product ID", "Product Name"],
        as_index=False
    )
    .agg(
        Revenue=("Sales", "sum"),
        Orders=("Order ID", "nunique")
    )
)

product_summary.head()

,Product ID,Product Name,Revenue,Orders
0,FUR-BO-10000112,"Bush Birmingham Collection Bookcase, Dark Cherry",825.174,1
1,FUR-BO-10000330,"Sauder Camden County Barrister Bookcase, Plank...",1064.624,3
2,FUR-BO-10000362,Sauder Inglewood Library Bookcases,2154.348,5
3,FUR-BO-10000468,O'Sullivan 2-Shelf Heavy-Duty Bookcases,723.842,6
4,FUR-BO-10000711,"Hon Metal Bookcases, Gray",851.760,2


In [19]:
product_validation = pd.DataFrame({
    "Validação": [
        "Produtos na visão analítica",
        "Product IDs distintos",
        "Receita da visão por produto",
        "Receita da base original",
        "Diferença de reconciliação"
    ],
    "Resultado": [
        f"{len(product_summary):,}",
        f"{product_summary['Product ID'].nunique():,}",
        f"${product_summary['Revenue'].sum():,.2f}",
        f"${df['Sales'].sum():,.2f}",
        f"${product_summary['Revenue'].sum() - df['Sales'].sum():,.2f}"
    ]
})

product_validation

,Validação,Resultado
0,Produtos na visão analítica,"1,893"
1,Product IDs distintos,"1,861"
2,Receita da visão por produto,"$2,261,255.41"
3,Receita da base original,"$2,261,255.41"
4,Diferença de reconciliação,$0.00


In [20]:
product_name_check = (
    df.groupby("Product ID")["Product Name"]
    .nunique()
    .sort_values(ascending=False)
)

product_name_conflicts = product_name_check[
    product_name_check > 1
]

print(
    "Product IDs associados a mais de um nome:",
    len(product_name_conflicts)
)

product_name_conflicts.head(10)

Product IDs associados a mais de um nome: 32


Product ID
OFF-AR-10001149    2
FUR-FU-10001473    2
FUR-FU-10004017    2
OFF-BI-10004632    2
FUR-FU-10004270    2
FUR-CH-10001146    2
FUR-FU-10004848    2
FUR-FU-10004864    2
OFF-AP-10000576    2
FUR-FU-10004091    2
Name: Product Name, dtype: int64

In [21]:
conflicting_product_ids = (
    product_name_conflicts
    .head(5)
    .index
)

df.loc[
    df["Product ID"].isin(conflicting_product_ids),
    ["Product ID", "Product Name"]
].drop_duplicates().sort_values(
    ["Product ID", "Product Name"]
)

,Product ID,Product Name
8417,FUR-FU-10001473,DAX Wood Document Frame
8432,FUR-FU-10001473,"Eldon Executive Woodline II Desk Accessories, ..."
8752,FUR-FU-10004017,"Executive Impressions 13"" Chairman Wall Clock"
8580,FUR-FU-10004017,Tenex Contemporary Contur Chairmats for Low an...
8418,FUR-FU-10004270,"Eldon Image Series Desk Accessories, Burgundy"
8514,FUR-FU-10004270,"Executive Impressions 13"" Clairmont Wall Clock"
3810,OFF-AR-10001149,Avery Hi-Liter Comfort Grip Fluorescent Highli...
3806,OFF-AR-10001149,"Sanford Colorific Colored Pencils, 12/Box"
700,OFF-BI-10004632,GBC Binding covers
200,OFF-BI-10004632,Ibico Hi-Tech Manual Binding System


#### Validação da Chave de Produto

A validação identificou 32 Product IDs associados a mais de um Product Name. A inspeção dos casos mostrou que essas associações representam produtos distintos, e não apenas variações textuais do mesmo nome.

Por esse motivo, Product ID isoladamente não será tratado como chave única nesta análise. A combinação **Product ID + Product Name** será utilizada como identificador analítico do produto, resultando em 1.893 combinações distintas.

A agregação mantém a integridade financeira da base, com receita reconciliada integralmente e diferença de $0,00 em relação aos dados processados.

In [22]:
product_metrics = (
    product_summary[
        ["Revenue", "Orders"]
    ]
    .describe()
    .loc[["min", "25%", "50%", "75%", "max"]]
    .round(2)
)

product_metrics["Revenue"] = (
    product_metrics["Revenue"]
    .map(lambda value: f"${value:,.2f}")
)

product_metrics["Orders"] = (
    product_metrics["Orders"]
    .map(lambda value: f"{value:,.0f}")
)

product_metrics

,Revenue,Orders
min,$1.62,1
25%,$94.86,3
50%,$279.96,5
75%,"$1,119.89",7
max,"$61,599.82",15


#### Interpretação

A distribuição evidencia forte heterogeneidade na contribuição econômica dos produtos.

Metade dos produtos gerou até aproximadamente $279,96 em receita e esteve presente em até 5 pedidos. No terceiro quartil, a receita acumulada por produto alcança cerca de $1,12 mil, com presença em até 7 pedidos.

No extremo superior, a maior receita observada chega a aproximadamente $61,60 mil, enquanto a maior presença registrada é de 15 pedidos. A diferença entre os valores centrais e os máximos sugere que uma parcela do portfólio possui contribuição econômica substancialmente superior àquela observada para o produto típico.

Esses resultados justificam a análise dos produtos de maior receita e, posteriormente, a avaliação do grau de concentração do faturamento no portfólio.

### 2.2 — Produtos de Maior Receita

O ranking por receita permite identificar os produtos com maior contribuição econômica acumulada.

Além da receita total, será considerada a presença em pedidos, permitindo verificar se os produtos de maior valor também apresentam maior recorrência nas transações.

In [23]:
top_products = (
    product_summary
    .sort_values("Revenue", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top_products_display = top_products.copy()

top_products_display["Revenue"] = (
    top_products_display["Revenue"]
    .map(lambda value: f"${value:,.2f}")
)

top_products_display.index = (
    top_products_display.index + 1
)

top_products_display

,Product ID,Product Name,Revenue,Orders
1,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,"$61,599.82",5
2,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,"$27,453.38",10
3,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,"$22,638.48",1
4,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,"$21,870.58",8
5,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,"$19,823.48",11
6,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,"$19,024.50",9
7,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,"$18,839.69",8
8,TEC-MA-10001127,HP Designjet T520 Inkjet Large Format Printer ...,"$18,374.90",3
9,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,"$17,965.07",6
10,OFF-SU-10000151,High Speed Automatic Electric Letter Opener,"$17,030.31",3


#### Interpretação

O ranking dos produtos de maior receita evidencia diferentes mecanismos de geração de valor.

O **Canon imageCLASS 2200 Advanced Copier** lidera o portfólio com aproximadamente $61,60 mil em receita, apesar de estar presente em apenas 5 pedidos. O **Cisco TelePresence System EX90** apresenta comportamento ainda mais extremo em termos de recorrência: ocupa a terceira posição, com cerca de $22,64 mil, associado a apenas 1 pedido.

Em contraste, produtos como o **Fellowes PB500 Electric Punch Plastic Comb Binding Machine** (10 pedidos) e o **GBC DocuBind TL300 Electric Binding System** (11 pedidos) alcançam elevada receita com maior presença nas transações.

O ranking confirma que os produtos de maior receita não dependem de um único padrão de recorrência: alguns alcançam elevada contribuição econômica por meio de poucas transações de alto valor, enquanto outros combinam maior presença em pedidos com receita acumulada elevada.

## 3. Concentração da Receita por Produto

### 3.1 — Participação dos Produtos na Receita

Após identificar os produtos de maior valor, a análise passa a avaliar o grau de concentração da receita no portfólio.

O objetivo é verificar qual parcela da receita total é gerada pelos produtos economicamente mais relevantes, permitindo distinguir um portfólio amplamente distribuído de uma estrutura mais concentrada em poucos produtos.

In [24]:
product_concentration = (
    product_summary
    .sort_values("Revenue", ascending=False)
    .reset_index(drop=True)
    .copy()
)

total_product_revenue = product_concentration["Revenue"].sum()

product_concentration["Revenue_Share"] = (
    product_concentration["Revenue"]
    / total_product_revenue
)

product_concentration["Cumulative_Revenue_Share"] = (
    product_concentration["Revenue_Share"].cumsum()
)

product_concentration["Product_Share"] = (
    (product_concentration.index + 1)
    / len(product_concentration)
)

### 3.2 — Concentração entre os Produtos de Maior Valor

A concentração será avaliada pela participação na receita dos 10% e 20% de produtos com maior faturamento acumulado.

Essa abordagem permite quantificar quanto da receita está associada às parcelas economicamente mais relevantes do portfólio.

In [25]:
top_10 = calculate_top_concentration(
    product_concentration["Revenue"],
    0.10
)

top_20 = calculate_top_concentration(
    product_concentration["Revenue"],
    0.20
)

top_10_count = top_10["top_count"]
top_10_revenue = top_10["top_value"]
top_10_share = top_10["top_share"]

top_20_count = top_20["top_count"]
top_20_revenue = top_20["top_value"]
top_20_share = top_20["top_share"]

In [26]:
concentration_summary = pd.DataFrame({
    "Faixa": ["Top 10%", "Top 20%"],
    "Produtos": [top_10_count, top_20_count],
    "Receita": [
        f"${top_10_revenue:,.2f}",
        f"${top_20_revenue:,.2f}"
    ],
    "Participação na Receita": [
        f"{top_10_share:.2%}",
        f"{top_20_share:.2%}"
    ]
})

concentration_summary

,Faixa,Produtos,Receita,Participação na Receita
0,Top 10%,190,"$1,336,554.93",59.11%
1,Top 20%,379,"$1,751,105.68",77.44%


#### Interpretação

A receita apresenta concentração elevada entre os produtos de maior valor econômico.

Os 10% de produtos com maior receita, correspondentes a 190 produtos, geraram aproximadamente $1,34 milhão, representando 59,11% da receita total. Ao ampliar a análise para os 20% superiores (379 produtos), a participação alcança 77,44%, equivalente a aproximadamente $1,75 milhão.

Consequentemente, os 80% restantes do portfólio respondem por apenas 22,56% da receita. A geração de faturamento está, portanto, fortemente concentrada em uma parcela relativamente pequena dos produtos analisados.

Esse resultado mostra que a relevância econômica do portfólio é bastante desigual e reforça a importância de investigar quais produtos estão associados às transações de maior impacto.

## 4. Produtos em Pedidos de Alto Valor

### 4.1 — Identificação dos Pedidos de Alto Valor

Para avaliar quais produtos estão associados às transações de maior impacto econômico, é necessário primeiro definir o valor total de cada pedido.

Como a base possui granularidade de item, a receita será consolidada por Order ID. A distribuição resultante será utilizada para estabelecer um critério objetivo de identificação dos pedidos de alto valor, evitando a adoção de um limite arbitrário.

In [27]:
order_summary = (
    df.groupby("Order ID", as_index=False)
    .agg(
        Order_Value=("Sales", "sum")
    )
)

order_summary.head()

,Order ID,Order_Value
0,CA-2015-100006,377.970
1,CA-2015-100090,699.192
2,CA-2015-100293,91.056
3,CA-2015-100328,3.928
4,CA-2015-100363,21.376


In [28]:
high_value_threshold = (
    order_summary["Order_Value"]
    .quantile(0.90)
)

high_value_orders = order_summary[
    order_summary["Order_Value"] >= high_value_threshold
].copy()

high_value_order_share = (
    len(high_value_orders)
    / len(order_summary)
)

high_value_revenue_share = (
    high_value_orders["Order_Value"].sum()
    / order_summary["Order_Value"].sum()
)

print(f"Limite de alto valor: ${high_value_threshold:,.2f}")
print(f"Pedidos de alto valor: {len(high_value_orders):,}")
print(f"Participação nos pedidos: {high_value_order_share:.2%}")
print(f"Participação na receita: {high_value_revenue_share:.2%}")

Limite de alto valor: $1,158.42
Pedidos de alto valor: 493
Participação nos pedidos: 10.02%
Participação na receita: 53.14%


#### Interpretação

O percentil 90 estabelece em aproximadamente $1.158,42 o limite para classificação dos pedidos de alto valor.

Com esse critério, 493 pedidos, equivalentes a 10,02% das transações, concentram 53,14% de toda a receita. Isso evidencia forte assimetria econômica no nível de pedido: uma parcela relativamente pequena das transações responde por mais da metade do faturamento.

O resultado reforça a relevância de identificar quais produtos estão associados a essas transações de maior impacto econômico.

### 4.2 — Produtos Associados às Transações de Maior Impacto

Após identificar os pedidos pertencentes aos 10% superiores por valor, a análise passa a avaliar quais produtos aparecem com maior frequência nessas transações.

O objetivo é identificar produtos recorrentes entre os pedidos de maior impacto econômico, distinguindo essa perspectiva do ranking geral de produtos por receita.

In [29]:
high_value_order_ids = set(
    high_value_orders["Order ID"]
)

high_value_items = df[
    df["Order ID"].isin(high_value_order_ids)
].copy()

In [30]:
high_value_products = (
    high_value_items
    .groupby(
        ["Product ID", "Product Name"],
        as_index=False
    )
    .agg(
        High_Value_Orders=("Order ID", "nunique"),
        Revenue_in_High_Value_Orders=("Sales", "sum")
    )
    .sort_values(
        ["High_Value_Orders", "Revenue_in_High_Value_Orders"],
        ascending=[False, False]
    )
    .head(10)
    .reset_index(drop=True)
)

high_value_products["Revenue_in_High_Value_Orders"] = (
    high_value_products["Revenue_in_High_Value_Orders"]
    .map(lambda value: f"${value:,.2f}")
)

high_value_products.index = (
    high_value_products.index + 1
)

high_value_products

,Product ID,Product Name,High_Value_Orders,Revenue_in_High_Value_Orders
1,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,10,"$19,285.28"
2,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,8,"$25,674.00"
3,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,8,"$18,263.52"
4,TEC-AC-10003033,Plantronics CS510 - Over-the-Head monaural Wir...,8,"$9,898.50"
5,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,7,"$20,749.01"
6,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,7,"$17,879.70"
7,FUR-CH-10001215,Global Troy Executive Leather Low-Back Tilter,7,"$11,472.44"
8,FUR-TA-10003473,Bretford Rectangular Conference Table Tops,7,"$9,798.19"
9,TEC-AC-10002049,Plantronics Savi W720 Multi-Device Wireless He...,7,"$9,367.29"
10,OFF-AP-10002945,Honeywell Enviracaire Portable HEPA Air Cleane...,7,"$8,899.24"


#### Interpretação

A análise mostra que alguns produtos apresentam presença recorrente entre as transações de maior impacto econômico.

O **GBC DocuBind TL300 Electric Binding System** aparece em 10 pedidos de alto valor, liderando o ranking, seguido pelo **Fellowes PB500 Electric Punch Plastic Comb Binding Machine** e pelo **GBC Ibimaster 500 Manual ProClick Binding System**, ambos presentes em 8 dessas transações.

A comparação com o ranking geral também revela sobreposição entre produtos de elevada receita e produtos recorrentes em pedidos de alto valor. O GBC DocuBind TL300, o Fellowes PB500, o HON 5400 Series Task Chairs e o Hewlett Packard LaserJet 3310 Copier aparecem nas duas perspectivas.

Os resultados indicam que parte dos produtos economicamente relevantes não apenas acumula elevada receita, mas também participa repetidamente das transações que concentram maior valor financeiro.

## 5. Síntese da Análise de Produtos

### 5.1 — Principais Achados

A análise no nível de produto mostrou forte desigualdade na contribuição econômica do portfólio e diferentes mecanismos de geração de receita.

- **A contribuição econômica varia significativamente entre os produtos.** A receita mediana por produto é de aproximadamente $279,96, enquanto o maior valor observado alcança $61,60 mil, evidenciando grande distância entre o produto típico e os produtos de maior receita.

- **Os produtos de maior receita não apresentam um único padrão de recorrência.** Alguns acumulam elevado faturamento com presença em poucos pedidos, enquanto outros combinam maior recorrência com receita elevada.

- **A receita está fortemente concentrada em uma parcela relativamente pequena do portfólio.** Os 10% de produtos de maior valor respondem por 59,11% da receita, enquanto os 20% superiores concentram 77,44%.

- **Os pedidos de alto valor possuem peso desproporcional no faturamento.** Apenas 10,02% dos pedidos, definidos pelo percentil 90 e com valor mínimo aproximado de $1.158,42, concentram 53,14% da receita total.

- **Alguns produtos apresentam presença recorrente nas transações de maior impacto.** O GBC DocuBind TL300 aparece em 10 pedidos de alto valor, enquanto o Fellowes PB500 e o GBC Ibimaster 500 aparecem em 8, mostrando sobreposição entre produtos de elevada relevância econômica e transações de maior valor.

Em conjunto, os resultados mostram que a geração de receita não está distribuída uniformemente pelo portfólio. Uma parcela relativamente pequena dos produtos concentra grande parte do faturamento e alguns desses produtos também apresentam presença recorrente nos pedidos economicamente mais relevantes.